# CV Matcher - LoRA Fine-Tuning with Gemma-2B

Bu notebook, **Gemma-2B** modelini CV parsing ve CV-JD matching için **LoRA** ile fine-tune eder.

## Kullanım
1. **Runtime → Change runtime type** → **T4 GPU** seç
2. **Bu hücreleri sırayla çalıştır** (Ctrl+F9)
3. Fine-tuning bittiğinde adapter'lar otomatik indirilir

---

### ⚠️ ÖNEMLİ: HuggingFace Lisansı
Gemma-2B gated bir modeldir. **Kullanmadan önce lisansı kabul etmelisin:**
1. https://huggingface.co/google/gemma-2b-it adresine git
2. "Agree and access repository" butonuna tıkla (giriş yapman gerekebilir)
3. https://huggingface.co/settings/tokens adresinden **Access Token** oluştur (Read role yeterli)
4. Aşağıdaki hücrede token'ı gir

## Adım 1: GPU Kontrolü ve Kütüphaneler

In [ ]:
from huggingface_hub import login
from getpass import getpass

print("HuggingFace token'ını gir (https://huggingface.co/settings/tokens):")
token = getpass()
login(token, add_to_git_credential=True)
print("Login başarılı!")

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
!pip install -q transformers accelerate peft trl datasets bitsandbytes scipy sentencepiece

## Adım 2: Projeyi ve Dataset'i Yükle

In [ ]:
!git clone https://github.com/ruveydagundogan/cvmatcher.git
%cd cvmatcher

import json
with open("backend/finetune/data/cv_parse_dataset.json") as f:
    cv_data = json.load(f)
with open("backend/finetune/data/cv_jd_match_dataset.json") as f:
    match_data = json.load(f)
print(f"CV Parse: {len(cv_data)} ornek")
print(f"CV-JD Match: {len(match_data)} ornek")

## Adım 3: CV Parsing Modeli Fine-Tune

Gemma-2B'yi CV'lerden skill/experience/education çıkarması için eğitiyoruz.

**Hiperparametreler:**
- LoRA rank (r): 8
- LoRA alpha: 16
- Epoch: 5
- Batch size: 4
- 4-bit quantization: True (VRAM için)

In [ ]:
import sys
sys.path.append("backend/finetune")

!python backend/finetune/train_lora.py \
    --base-model google/gemma-2b-it \
    --data backend/finetune/data/cv_parse_dataset.json \
    --output-dir /content/cvmatcher-lora/cv-parser-v1 \
    --mode cv-parse \
    --epochs 5 \
    --batch-size 4 \
    --max-length 512 \
    --lr 2e-4 \
    --quantize

print("CV Parse modeli fine-tune edildi!")

## Adım 4: CV-JD Matching Modeli Fine-Tune

In [ ]:
!python backend/finetune/train_lora.py \
    --base-model google/gemma-2b-it \
    --data backend/finetune/data/cv_jd_match_dataset.json \
    --output-dir /content/cvmatcher-lora/cv-jd-matcher-v1 \
    --mode cv-jd-match \
    --epochs 5 \
    --batch-size 4 \
    --max-length 512 \
    --lr 2e-4 \
    --quantize

print("CV-JD Match modeli fine-tune edildi!")

## Adım 5: Base Model vs Fine-Tuned Model Karşılaştırması

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel

test_cv = """Python backend developer with 4 years experience.
Skilled in Django, FastAPI, PostgreSQL, Redis, Celery, Docker.
Built REST APIs serving 50K requests per minute.
Bachelor's in Software Engineering."""

prompt = f"""<start_of_turn>user
Parse the following CV text and extract structured information: skills, experience, education, and a brief summary.

{test_cv}
<end_of_turn>
<start_of_turn>model
"""

print("=" * 60)
print("BASE MODEL (Gemma-2B) TESTI")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")
base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b-it",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = base_model.generate(**inputs, max_new_tokens=256, temperature=0.1)
base_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(base_response[-500:] if len(base_response) > 500 else base_response)

In [ ]:
print("=" * 60)
print("FINE-TUNED MODEL (LoRA) TESTI")
print("=" * 60)

finetuned = PeftModel.from_pretrained(
    base_model, "/content/cvmatcher-lora/cv-parser-v1"
)
finetuned.eval()

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = finetuned.generate(**inputs, max_new_tokens=256, temperature=0.1)
ft_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(ft_response[-500:] if len(ft_response) > 500 else ft_response)

## Adım 6: Adapter'ları İndir

Fine-tune edilen LoRA adapter'larını bilgisayarına indir.

In [ ]:
import zipfile
import os

zip_path = "/content/cvmatcher-lora.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk("/content/cvmatcher-lora"):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, "/content/cvmatcher-lora")
            zf.write(file_path, arcname)

print(f"Zip olu\u015fturuldu: {zip_path}")
print("\u0130ndirmek i\u00e7in sol paneldeki dosya ikonuna t\u0131kla \u2192 cvmatcher-lora.zip \u2192 ... \u2192 Download")
print("\nAlternatif:")
from google.colab import files
files.download(zip_path)

## Adım 7: Ollama'ya Yükleme (Mac'te)

```bash
# indirdigin zip'i ac
unzip ~/Downloads/cvmatcher-lora.zip -d backend/finetune/adapters/

# Ollama custom model olustur
ollama create cv-parser -f backend/finetune/adapters/cv-parser-v1/Modelfile
ollama create cv-jd-matcher -f backend/finetune/adapters/cv-jd-matcher-v1/Modelfile

# Test et
ollama run cv-parser
```

### Backend'de kullanmak için:
```bash
export OLLAMA_MODEL=cv-parser
go run cmd/server/main.go
```